In [1]:
import os

base = r'C:\Users\neo62\sperm-ai\data\raw\VISEM-Tracking\VISEM_Tracking_Train_v4\Train'

# 전체 참가자 목록과 이미지/라벨 수 확인
participants = sorted(os.listdir(base))
print(f"총 참가자 수: {len(participants)}명")
print(f"참가자 ID: {participants}\n")

total_images = 0
for p in participants:
    img_path = os.path.join(base, p, 'images')
    lbl_path = os.path.join(base, p, 'labels')
    if os.path.exists(img_path):
        n_imgs = len(os.listdir(img_path))
        n_lbls = len(os.listdir(lbl_path))
        total_images += n_imgs
        print(f"참가자 {p:>3}: 이미지 {n_imgs:>5}장, 라벨 {n_lbls:>5}개")

print(f"\n전체 이미지 수: {total_images:,}장")

총 참가자 수: 20명
참가자 ID: ['11', '12', '13', '14', '15', '19', '21', '22', '23', '24', '29', '30', '35', '36', '38', '47', '52', '54', '60', '82']

참가자  11: 이미지  1470장, 라벨  1470개
참가자  12: 이미지  1470장, 라벨  1470개
참가자  13: 이미지  1470장, 라벨  1470개
참가자  14: 이미지  1470장, 라벨  1470개
참가자  15: 이미지  1470장, 라벨  1470개
참가자  19: 이미지  1470장, 라벨  1470개
참가자  21: 이미지  1470장, 라벨  1470개
참가자  22: 이미지  1470장, 라벨  1470개
참가자  23: 이미지  1470장, 라벨  1296개
참가자  24: 이미지  1470장, 라벨  1470개
참가자  29: 이미지  1470장, 라벨  1470개
참가자  30: 이미지  1470장, 라벨  1470개
참가자  35: 이미지  1440장, 라벨  1440개
참가자  36: 이미지  1470장, 라벨  1470개
참가자  38: 이미지  1470장, 라벨  1470개
참가자  47: 이미지  1470장, 라벨  1470개
참가자  52: 이미지  1440장, 라벨  1440개
참가자  54: 이미지  1470장, 라벨  1470개
참가자  60: 이미지  1470장, 라벨  1470개
참가자  82: 이미지  1500장, 라벨  1500개

전체 이미지 수: 29,370장


In [2]:
import os
import shutil
import random
from pathlib import Path

random.seed(42)

# 경로 설정
base = r'C:\Users\neo62\sperm-ai\data\raw\VISEM-Tracking\VISEM_Tracking_Train_v4\Train'
output = r'C:\Users\neo62\sperm-ai\data\processed\yolo_dataset'

# 출력 폴더 생성
for split in ['train', 'val']:
    Path(f'{output}/images/{split}').mkdir(parents=True, exist_ok=True)
    Path(f'{output}/labels/{split}').mkdir(parents=True, exist_ok=True)

# 참가자 분할 (16명 train, 4명 val)
participants = sorted(os.listdir(base))
random.shuffle(participants)
train_p = participants[:16]
val_p   = participants[16:]

print(f"Train 참가자 ({len(train_p)}명): {sorted(train_p)}")
print(f"Val   참가자 ({len(val_p)}명): {sorted(val_p)}")

# 파일 복사
def copy_files(participant_list, split):
    count = 0
    skipped = 0
    for p in participant_list:
        img_dir = os.path.join(base, p, 'images')
        lbl_dir = os.path.join(base, p, 'labels')
        
        for img_file in os.listdir(img_dir):
            if not img_file.endswith('.jpg'):
                continue
            
            # 라벨 파일명 매칭
            lbl_file = img_file.replace('.jpg', '.txt')
            lbl_path = os.path.join(lbl_dir, lbl_file)
            
            # 라벨 없으면 스킵 (참가자 23 처리)
            if not os.path.exists(lbl_path):
                skipped += 1
                continue
            
            # 파일명 충돌 방지: 참가자번호_파일명
            new_name = f"{p}_{img_file}"
            
            shutil.copy(
                os.path.join(img_dir, img_file),
                f'{output}/images/{split}/{new_name}'
            )
            shutil.copy(
                lbl_path,
                f'{output}/labels/{split}/{new_name.replace(".jpg", ".txt")}'
            )
            count += 1
    
    print(f"{split}: {count}장 복사 완료, {skipped}장 스킵")
    return count

train_count = copy_files(train_p, 'train')
val_count   = copy_files(val_p,   'val')

print(f"\n전체: {train_count + val_count}장")
print(f"Train: {train_count}장 / Val: {val_count}장")

Train 참가자 (16명): ['12', '13', '15', '19', '21', '24', '29', '30', '35', '36', '38', '47', '52', '54', '60', '82']
Val   참가자 (4명): ['11', '14', '22', '23']
train: 23490장 복사 완료, 0장 스킵
val: 5706장 복사 완료, 174장 스킵

전체: 29196장
Train: 23490장 / Val: 5706장


In [3]:
yaml_content = """# VISEM-Tracking Dataset for YOLO11
path: C:/Users/neo62/sperm-ai/data/processed/yolo_dataset
train: images/train
val: images/val

# 클래스
nc: 3
names:
  0: sperm
  1: cluster
  2: small
"""

yaml_path = r'C:\Users\neo62\sperm-ai\data\processed\yolo_dataset\visem.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("✅ visem.yaml 생성 완료")
print(f"경로: {yaml_path}")
print("\n내용:")
print(yaml_content)

✅ visem.yaml 생성 완료
경로: C:\Users\neo62\sperm-ai\data\processed\yolo_dataset\visem.yaml

내용:
# VISEM-Tracking Dataset for YOLO11
path: C:/Users/neo62/sperm-ai/data/processed/yolo_dataset
train: images/train
val: images/val

# 클래스
nc: 3
names:
  0: sperm
  1: cluster
  2: small

